# Ranh giới LEZ-first: mốc 2027 và 2030

**A = vùng thí điểm LEZ 2027; B = LEZ 2030 trừ A; C = Hà Nội ngoài LEZ 2030.** Bộ 60 là A20/B20/C20; bộ 30 là A10/B8/C12.

Nguồn phê duyệt hiện có quy định năm 2027 tại KV1/KV2/KV3 (Hoàn Kiếm, Cửa Nam). Toàn bộ Vành đai 1 áp dụng từ 2028; lớp Vành đai 1 được giữ để đối chiếu. Hình học đang dùng là proxy kỹ thuật theo đường OSM và bản đồ nguồn, `official_gis_verified=False`.

Notebook chạy bằng Python thường hoặc kernel PyQGIS. Phần dựng hình gọi runtime đã đăng ký và chỉ ghi vào thư mục tạm. Không gửi HTTP, không ghi đầu ra chính thức hoặc sinh/chọn tuyến. Restart Kernel sau khi cập nhật module.


In [ ]:
from pathlib import Path
import sys, json, os, tempfile, subprocess
import pandas as pd
from IPython.display import display, Image

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates
             if (p / "scripts/lez/policy_boundaries.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Không tìm thấy work đã cài policy_boundaries.py.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.pipeline import kernel_spec
CURRENT_POLICY = json.loads((WORK / "config/lez_policy.json").read_text())
SHOW_HISTORICAL = False
print("Dự án:", WORK)
print("Phiên bản LEZ-first:", CURRENT_POLICY["policy_version"])


## 1. Kiểm tra mốc chính sách và nguồn

Đọc cấu hình LEZ-first đã đóng băng. Dân cư/hoạt động không điều chỉnh A/B/C. Nguồn pháp lý có ngày và hash; hình học GIS chính thức vẫn cần xác minh.


In [ ]:
display(pd.DataFrame([
    {"nhóm": "A", "định_nghĩa": "KV1/KV2/KV3 thí điểm 2027", "ứng_viên": 20, "chọn_30": 10},
    {"nhóm": "B", "định_nghĩa": "LEZ 2030 trừ A", "ứng_viên": 20, "chọn_30": 8},
    {"nhóm": "C", "định_nghĩa": "Hà Nội ngoài LEZ 2030", "ứng_viên": 20, "chọn_30": 12},
]))
display(pd.DataFrame(CURRENT_POLICY["sources"]).reindex(columns=[
    "source_id", "published_date", "file", "sha256", "url"
]))
print("Cho phép điều chỉnh theo dân cư/hoạt động:",
      CURRENT_POLICY["allow_population_activity_boundary_adjustments"])
print("Mốc 2027:", CURRENT_POLICY["statutory2027"])
print("Mốc 2030:", CURRENT_POLICY["statutory2030"])


## 2. Dựng và kiểm tra trong thư mục tạm

Dùng hình học OSM đã lưu và hash nguồn. Kiểm tra A nằm trong 2030, A/B/C không chồng lấn và phủ Hà Nội. Dải hình học chưa chắc chắn được lưu riêng và loại khỏi khả năng sinh tuyến.

`RUN_POLICY_PREVIEW=True` dựng lớp tạm; đổi thành `False` để chỉ đọc cấu hình và nguồn.


In [ ]:
WORKER_SOURCE = 'import json, sys\nfrom pathlib import Path\nwork, folder = map(Path, sys.argv[1:3])\nsys.path.insert(0, str(work / "scripts"))\nfrom lez.policy_boundaries import build, configuration\nfrom lez.gis import initialize, read_polygon\nfrom lez.common import input_path\nfrom qgis.core import (QgsGeometry, QgsVectorLayer, QgsMapSettings, QgsMapRendererParallelJob,\n                       QgsCategorizedSymbolRenderer, QgsRendererCategory, QgsFillSymbol)\nfrom qgis.PyQt.QtCore import QSize\nfrom qgis.PyQt.QtGui import QColor\ncontext = initialize(folder / "runtime")\nboundary = json.loads((work / "config/boundary_project.json").read_text())\nhanoi = read_polygon(input_path(work, boundary["hanoi_source"]["path"]), "EPSG:3405", context)\nresult = build(work, hanoi, context, output_dir=folder, options=configuration(work))\nlayer = QgsVectorLayer(str(folder / "policy_boundaries.gpkg") + "|layername=abc_groups", "LEZ-first", "ogr")\ncolors = {"A":"#9c2b88", "B":"#269b8e", "C":"#e5d5b5"}\nlayer.setRenderer(QgsCategorizedSymbolRenderer("abc_group", [\n    QgsRendererCategory(k, QgsFillSymbol.createSimple({"color":v,"outline_color":"#364152","outline_width":"0.2"}), k)\n    for k,v in colors.items()]))\nfor name, geometry in (("lez2030",result["lez2030"]),("pilot2027",result["lez2027"])):\n    settings=QgsMapSettings();settings.setLayers([layer]);settings.setDestinationCrs(layer.crs())\n    extent=geometry.boundingBox();extent.scale(1.15);settings.setExtent(extent)\n    settings.setBackgroundColor(QColor("white"));settings.setOutputSize(QSize(1100,850))\n    job=QgsMapRendererParallelJob(settings);job.start();job.waitForFinished()\n    if not job.renderedImage().save(str(folder/(name+".png")),"PNG"):\n        raise RuntimeError("Cannot save temporary boundary map")\nprint(json.dumps({"directory":str(folder),"report":str(folder/"policy_boundaries_report.json")},ensure_ascii=False))\n'
RUN_POLICY_PREVIEW = True
POLICY_REPORT = None
TEMP_OUTPUT = None
if RUN_POLICY_PREVIEW:
    TEMP_OUTPUT = Path(tempfile.mkdtemp(prefix="hanoi_lez_first_boundary_"))
    worker = TEMP_OUTPUT / "preview_worker.py"
    worker.write_text(WORKER_SOURCE, encoding="utf-8")
    spec = kernel_spec()
    env = os.environ.copy(); env.update(spec.get("env", {}))
    proc = subprocess.run([spec["argv"][0], str(worker), str(WORK), str(TEMP_OUTPUT)],
                          env=env, text=True, capture_output=True, timeout=300)
    if proc.returncode:
        raise RuntimeError("Kiểm tra ranh giới thất bại:\n" + proc.stdout[-2000:] + proc.stderr[-2000:])
    POLICY_REPORT = json.loads((TEMP_OUTPUT / "policy_boundaries_report.json").read_text())
    print("Thư mục tạm:", TEMP_OUTPUT)
    print("Trạng thái:", POLICY_REPORT["status"])
    print("GIS chính thức đã xác minh:", POLICY_REPORT["official_gis_verified"])


## 3. Diện tích, bản đồ và điểm chưa chắc chắn

A màu tím; B màu xanh; C màu be. Bản đồ đầu phóng theo vùng 2030; bản đồ sau phóng theo vùng thí điểm 2027. Đây là bản đồ proxy kỹ thuật. Tổng diện tích A/B/C và khoảng hở/chồng lấn được kiểm tra trên toàn Hà Nội.


In [ ]:
if POLICY_REPORT is not None:
    display(pd.DataFrame([{"nhóm": k, "diện_tích_km2": v}
                          for k, v in POLICY_REPORT["group_areas_km2"].items()]))
    display(pd.DataFrame([{"kiểm_tra": k, "kết_quả": v}
                          for k, v in POLICY_REPORT["checks"].items()]))
    print("Lớp GIS tạm:", TEMP_OUTPUT / "policy_boundaries.gpkg")
    display(Image(filename=str(TEMP_OUTPUT / "lez2030.png")))
    display(Image(filename=str(TEMP_OUTPUT / "pilot2027.png")))
    display(pd.DataFrame([POLICY_REPORT["historical_admin_reference_comparison"]]))
    print("Dải loại khỏi sinh tuyến:", POLICY_REPORT["routing_exclusion"])
    print("Các đường trong mô tả nguồn chưa có trên đường bao proxy:",
          POLICY_REPORT["pilot2027"]["declared_official_streets_absent_from_final_trace"])
    for issue in POLICY_REPORT["pilot2027"]["interpretation_issues"]:
        print("-", issue)


## 4. Chuyển sang sinh tuyến

Notebook 05 sử dụng bộ ranh giới LEZ-first; notebook 06 chọn A10/B8/C12. A×RC1 không có phần đường đủ điều kiện định tuyến sau dải loại trừ: phân bổ 20 mục tiêu A cho RC2/RC3/RC4 theo 6/7/7. Ma trận vẫn giữ đủ 12 ô A/B/C × RC1–RC4; chiều dài mạng cắt theo nhóm và mẫu số định tuyến thuần nhóm là hai đại lượng riêng.

Các kết luận về quyền đi, công trình, tính lặp và điểm dừng an toàn vẫn cần chứng cứ thực địa. Phần dưới chỉ dùng đối chiếu phân tầng V/N và phạm vi điều chỉnh cũ, không xác định A/B/C hiện hành.


# Đối chiếu lịch sử: phân tầng V/N và phạm vi theo dân cư/hoạt động

`SHOW_HISTORICAL=False` mặc định bỏ qua toàn bộ code dưới đây. Nếu bật, các bước cũ có thể ghi lại báo cáo/lớp dẫn xuất lịch sử; chúng không được dùng làm phạm vi LEZ-first hiện hành. Phần này giữ các ô nguồn, bản đồ và truy vết cũ phục vụ so sánh.


In [ ]:
if SHOW_HISTORICAL:
    from pathlib import Path
    import sys
    import json
    import pandas as pd
    from IPython.display import display, Image

    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [p / "work" for p in candidates]
    candidates += [Path.home() / "python" / "work"]
    WORK = next((p.resolve() for p in candidates
                 if (p / "scripts/lez/normalization_2030.py").is_file()), None)
    if WORK is None:
        raise FileNotFoundError("Không tìm thấy folder work có chuẩn hóa LEZ 2030.")
    if str(WORK / "scripts") not in sys.path:
        sys.path.insert(0, str(WORK / "scripts"))

    from lez.sampling_2030 import preflight_sampling
    from lez.normalization_2030 import run_normalization, normalization_status
    from lez.boundary_review import review_normalization, render_current_boundaries
    from lez.common import input_path


### Lịch sử — 1. Chính sách đã dùng cho nghiên cứu 2030

Đọc `work/config/sampling_boundary_policy.json` và `work/config/sampling_2030.json`. Ranh giới đối chiếu là hợp 36 địa bàn theo tên trong văn bản với hình học OSM; chưa xác minh là GIS chính thức. Phạm vi lấy mẫu được lưu riêng, cho phép điều chỉnh các phần trong dải sát ranh giới.

Các gán bổ sung trong cấu hình là vùng tham chiếu ban đầu; sau khi cắt theo phạm vi lấy mẫu, từng phần có thể được chuyển sang tầng V/N phù hợp. Bảng truy vết ở bước 5 thể hiện phân tầng cuối cùng.


In [ ]:
if SHOW_HISTORICAL:
    policy = json.loads((WORK / "config/sampling_boundary_policy.json").read_text())
    options = json.loads((WORK / "config/sampling_2030.json").read_text())
    display(pd.DataFrame([{
        "mốc_nghiên_cứu": policy["legal_reference"]["scope_selection_status"],
        "hình_học_đối_chiếu": policy["legal_reference"]["geometry_status"],
        "dải_mỗi_phía_m": options["boundary_band_m"],
        "trọng_số_dân_cư": options["classification_weights"]["population_density"],
        "trọng_số_hoạt_động": options["classification_weights"]["activity_density"],
        "ngưỡng_vào_trong": options["score_threshold"],
    }]))
    print("Các gán bổ sung trong cấu hình là vùng tham chiếu, không phải nhóm của toàn địa bàn sau khi cắt.")


### Lịch sử — 2. Kiểm tra đầu vào và runtime — không gửi HTTP

Kiểm tra cấu hình, snapshot, nguồn pháp lý đã lưu và runtime PyQGIS. Kết quả chuẩn hóa/cache được kiểm tra lại khi chạy bước 3; có dữ liệu thiếu hoặc hash sai thì báo lỗi rõ ràng, không tự quay về phương án cũ.


In [ ]:
if SHOW_HISTORICAL:
    CHECK = preflight_sampling(WORK, progress=print)
    display(pd.DataFrame([{key: CHECK.get(key) for key in (
        "scope_selection", "snapshot_utc", "network_calls", "raw_osm_modified",
        "reference_gis_verified", "research_scope_only")
    }]))


### Lịch sử — 3. Chạy hoặc tiếp tục chuẩn hóa ranh giới/phân tầng

`fetch=False` dùng dân cư/hoạt động đã lưu; `force=False` đọc cache đúng cấu hình và hash. Chạy lại cell này để tiếp tục sau khi dừng. Bộ chuẩn hóa không sinh tuyến hoặc lựa chọn 30 tuyến.

Nếu nguồn còn thiếu, xử lý đầu vào theo hướng dẫn ở notebook 05. Không tự bật tải trong notebook kiểm tra ranh giới.


In [ ]:
if SHOW_HISTORICAL:
    A_RESULT = run_normalization(WORK, fetch=False, force=False, progress=print)
    REVIEW = review_normalization(WORK, A_RESULT)
    print("Trạng thái chuẩn hóa:", A_RESULT["status"])
    print("Phiên bản đang hiển thị:", REVIEW["normalization_version"])
    print("Phân tầng nghiên cứu đạt kiểm tra:", REVIEW["research_partition_ready"])
    print("GIS chính thức đã xác minh:", REVIEW["official_gis_verified"])
    print("Đọc cache:", A_RESULT.get("cache_hit", False))


### Lịch sử — 4. Tổng kết phân tầng lịch sử

Số địa bàn chưa gán được đối chiếu giữa danh sách địa bàn nguồn và bảng truy vết lịch sử. V1–V5 thuộc nhóm trong, N1–N10 thuộc nhóm ngoài của phạm vi lấy mẫu nghiên cứu. Một địa bàn bị cắt có thể xuất hiện trong nhiều tầng; không cộng cột số địa bàn của các tầng để suy ra tổng địa bàn.

Trạng thái GIS chính thức được báo riêng, không dùng `stage_a_complete` của cấu hình A/C cũ để kết luận phân tầng 2030 chưa hoàn thành.


In [ ]:
if SHOW_HISTORICAL:
    boundary = A_RESULT["boundary"]
    display(pd.DataFrame([{
        "phiên_bản": REVIEW["normalization_version"],
        "hoàn_chỉnh_lúc": REVIEW["finished_at"],
        "địa_bàn_nguồn": REVIEW["admin_units"],
        "số_tầng": len(REVIEW["strata"]),
        "địa_bàn_chưa_gán": len(REVIEW["unassigned_admin_units"]),
        "địa_bàn_gán_bổ_sung": len(REVIEW["remaining_assignments"]),
        "phần_sát_ranh_giới_đổi_nhóm": boundary["adjusted_units"],
        "phần_chờ_chứng_cứ": len(REVIEW["pending_evidence_parts"]),
        "phân_tầng_nghiên_cứu_đạt": REVIEW["research_partition_ready"],
        "GIS_chính_thức_đã_xác_minh": REVIEW["official_gis_verified"],
    }]))
    display(pd.DataFrame(REVIEW["strata"]))


### Lịch sử — 5. Bản đồ phân tầng lịch sử và truy vết địa bàn

Bản đồ đọc lớp `strata` trong GeoPackage chuẩn hóa lịch sử; bản đồ được gắn hash của lớp nguồn. Màu xanh dương là các tầng V, màu xanh lá là các tầng N. Nét đứt thể hiện đường bao đối chiếu 2030, không phải khẳng định GIS pháp lý đã được xác minh.

Bảng đầu ghi các gán bổ sung. Bảng sau ghi **từng phần** địa bàn với tầng tham chiếu, tầng lấy mẫu cuối cùng và lý do. `technical_gap` là phần vá kỹ thuật có truy vết, không phải một xã/phường mới.


In [ ]:
if SHOW_HISTORICAL:
    MAP_REPORT = render_current_boundaries(WORK, A_RESULT)
    display(Image(filename=str(input_path(WORK, MAP_REPORT["path"]))))
    print("QA hình học thực tế trên bản đồ:", MAP_REPORT["geometry_checks"])
    display(pd.DataFrame(REVIEW["remaining_assignments"]))
    display(pd.DataFrame(REVIEW["trace"]).reindex(columns=[
        "admin_id", "name", "reference_stratum", "sampling_stratum", "group", "area_m2", "reason"
    ]))


### Lịch sử — 6. Phần còn cần xử lý — tách theo loại

**Phân tầng lịch sử:** xem số địa bàn chưa gán, phần thiếu chứng cứ và từng kiểm tra. Bảng trống có nghĩa không còn mục thuộc loại đó trong phiên bản này.

**Hình học OSM gốc:** khoảng hở/chồng lấn vẫn được ghi nhận. Lớp nghiên cứu đã xử lý trên dữ liệu dẫn xuất; giữ dung sai kiểm tra 0,1 m² cho phân hoạch dẫn xuất và giữ nguyên QA nguồn.

**GIS chính thức:** tiếp tục chờ xác minh khi `official_gis_verified=False`. Đây là yêu cầu xác minh nguồn, không phải lỗi gán V/N.

`SHOW_LEGACY=False` ẩn bảng yêu cầu của cấu hình A/C cũ. Khi bật, bảng được ghi rõ là lịch sử/phương án cũ, không phải lỗi phân tầng 2030 lịch sử.


In [ ]:
if SHOW_HISTORICAL:
    display(pd.DataFrame([{"kiểm_tra_hiện_tại": key, "đạt": passed}
                          for key, passed in REVIEW["checks"].items()]))
    display(pd.DataFrame([
        {"dữ_liệu": "OSM địa bàn gốc", "khoảng_hở_m2": REVIEW["raw_admin_qa"]["missing_area_m2"],
         "chồng_lấn_m2": REVIEW["raw_admin_qa"]["overlap_area_m2"],
         "diễn_giải": "QA nguồn còn ghi nhận; nguồn gốc giữ nguyên"},
        {"dữ_liệu": "Phân tầng nghiên cứu hiện tại", "khoảng_hở_m2": REVIEW["derived_gap_m2"],
         "chồng_lấn_m2": REVIEW["derived_overlap_m2"],
         "diễn_giải": f"Dung sai kiểm tra dẫn xuất: {REVIEW['tolerance_m2']} m²"},
    ]))
    print("Địa bàn hiện tại chưa gán:", len(REVIEW["unassigned_admin_units"]))
    if REVIEW["unassigned_admin_units"]:
        display(pd.DataFrame(REVIEW["unassigned_admin_units"]))
    print("Phần hiện tại còn thiếu chứng cứ:", len(REVIEW["pending_evidence_parts"]))
    if REVIEW["pending_evidence_parts"]:
        display(pd.DataFrame(REVIEW["pending_evidence_parts"]))
    if REVIEW["issues"]:
        display(pd.DataFrame(REVIEW["issues"]))
    else:
        print("Không còn mục cần xử lý trong các kiểm tra này.")

    SHOW_LEGACY = False
    legacy_path = WORK / "reports/boundaries/decisions_required.csv"
    if SHOW_LEGACY and legacy_path.is_file():
        print("LỊCH SỬ/CẤU HÌNH A/C CŨ — không dùng bảng này để đánh giá phân tầng 2030 hiện tại.")
        display(pd.read_csv(legacy_path, keep_default_na=False))


### Lịch sử — 7. Nguồn và tệp GIS của đúng phiên bản

Mở lớp phân tầng lịch sử theo đường dẫn `zones` trong bảng dưới. `work/data/processed/sampling_2030.gpkg` trỏ đến đầu ra chuẩn hóa; `zones.gpkg` và `zones_draft.gpkg` thuộc quy trình A/C cũ.

Thông tin tải tài liệu bên dưới là nhật ký nguồn đã lưu, có thời điểm riêng. Một lần GET trang nguồn bị timeout không tự có nghĩa phân tầng nghiên cứu lịch sử bị lỗi. Notebook không tải lại trang nguồn.


In [ ]:
if SHOW_HISTORICAL:
    display(pd.DataFrame([{"loại": key, "tệp": value} for key, value in A_RESULT["paths"].items()]))
    print("Bản đồ hiện tại:", MAP_REPORT["path"])
    print("Tài liệu pháp lý đối chiếu:", policy["legal_reference"]["source_file"])
    latest = normalization_status(WORK)
    print("Chuẩn hóa mới nhất:", latest.get("finished_at"), latest.get("status"))
    source_log = WORK / "reports/boundaries/latest_stage_a.json"
    if source_log.is_file():
        historical = json.loads(source_log.read_text())
        print("Nhật ký tải nguồn của quy trình A/C cũ:", historical.get("started_at"))
        capture = historical.get("source_capture", {})
        source_rows = capture.get("html", []) + capture.get("pdf", [])
        if source_rows:
            display(pd.DataFrame(source_rows).reindex(columns=["source_id", "status", "local_file", "error"]))
